# http模块构建服务器提供api服务-练习题

## 练习注意事项：
**运行代码块之后，服务器是运行状态，在浏览器里测试服务器运行状况。
测试完成后，手动停止代码块，然后进行下一个练习。**



练习1：使用 http.server以及BaseHTTPRequestHandler 部署一个静态 html 服务器

测试方法：
- 在浏览器里访问: http://127.0.0.1:8010
- 尝试在URL后面添加一些字符，查看是否可以正常访问html页面。
比如：http://127.0.0.1:8000/good

In [ ]:
# import http.server
from http.server import HTTPServer, BaseHTTPRequestHandler

PORT = 8010

class CustomHandler(BaseHTTPRequestHandler):
    def do_GET(self):
        self.send_response(200)
        self.send_header("Content-type", "text/html")
        self.end_headers()
        html = """
        <!DOCTYPE html>
        <html lang="zh-CN">
        <head>
            <meta charset="UTF-8">
            <title>Python 服务器</title>
        </head>
        <body>
            <h1>Python HTTP Server</h1>
            <p>访问成功！端口：8010</p>
            <p>中文显示测试：你好，世界！</p>
        </body>
        </html>
        """
        self.wfile.write(html.encode())

with HTTPServer(("127.0.0.1", PORT), ) as httpd:  # TODO: 填写处理类
    print(f"服务器运行中，访问 http://localhost:{PORT}")
    httpd.serve_forever()

练习2：用SimpleHTTPRequestHandler构建一个文件服务器

测试方法：
- 在浏览器里打开 http://127.0.0.1:8010
- 请先停止服务器。然后尝试在命令行下，执行 
```
python -m http.server
``` 
与以上SimpleHTTPREquestHandler所实现的文件服务器效果进行对比。

In [ ]:
from http.server import SimpleHTTPRequestHandler, HTTPServer  
server = HTTPServer(('localhost', 8010), )  # TODO: 填写处理类
server.serve_forever()  

练习3：使用 http.server 部署一个返回 JSON数据 的 API 服务器
- 当访问 /api 时返回 application/json 的数据，例如: {"message":"hello","status":"ok"}
- 其他路径返回 404

测试方法：
- 在浏览器里访问 http://127.0.0.1:8001/api
- 在浏览器里访问 http://127.0.0.1:8001 或其他比如: http://127.0.0.1:8001/newapi
- 测试之前的美国气象局api URL： https://api.weather.gov/gridpoints/TOP/30,80/forecast ， 看是否也可以通过浏览器获取数据。
 

In [ ]:
from http.server import HTTPServer, BaseHTTPRequestHandler

PORT = 8001

class APIHandler(BaseHTTPRequestHandler):
  def do_GET(self):
    if self.path == "/api":
      data = b'{"message":"hello","status":"ok","items":[1,2,3]}'
      self.send_response(200)
      self.send_header("Content-Type", "application/json; charset=utf-8")
      self.send_header("Content-Length", str(len(data)))
      self.end_headers()
      self.wfile.write()  # TODO, 填写要发送的数据
    else:
      self.send_error(404)

if __name__ == "__main__":
  HTTPServer(("127.0.0.1", PORT), ).serve_forever()  # TODO: 填写处理类

练习4：使用 urllib.request 获取练习3里 的 json 服务器返回的内容
- 构造 GET 请求，读取响应并用 json.loads 解析

测试方法：
- 需要将练习3里的代码单独复制到一个 json-server.py 文件里，然后单独运行（在cmd下运行，或者在pycharmm里点击运行），然后执行下面代码块里的代码进行访问。
- 将 URL = "http://127.0.0.1:8020/api" 里的"/api"修改为其他内容，尝试是否还能访问。


In [ ]:
import json
import urllib.request

URL = "http://127.0.0.1:8001/api"

def main():
    resp = urllib.request.urlopen(URL)
    raw =  # TODO: 读取响应内容，raw为字符串类型
    print(raw)
    data = json.loads(raw)
    print(data["message"])

if __name__ == "__main__":
    main()

练习5：给第2步里的 json 服务器添加鉴权，鉴权 key 存储在一个变量里
- 在服务器端实现基于 Authorization: Bearer <key> 的鉴权
- 未通过鉴权时返回 401 与提示
- 通过鉴权后返回与练习4 相同的 JSON

测试方法：
- 运行代码块，不输出任何内容且不报错。

In [ ]:
import http.server

PORT = 8001

# 预定义允许的 keys
keys_lib = {"sk-123", "sk-321"}

class APIHandler(http.server.BaseHTTPRequestHandler):
    def _get_api_key_from_header(self):
        auth = self.headers.get("Authorization")
        if not auth:
            return None
        parts = auth.split()
        if len(parts) == 2 and parts[0].lower() == "bearer":
            return parts[1]
        return None

    def _key_allowed(self, key):
        if not key:
            return False
        return key in keys_lib

    def do_GET(self):
        # 鉴权：检查 headers 里的 Authorization 是否在 keys_lib 中
        api_key = self._get_api_key_from_header()
        if not self._key_allowed(api_key):
            data = "认证不通过".encode("utf-8")
            self.send_response(401)
            self.send_header("Content-Type", "text/plain; charset=utf-8")
            self.send_header("Content-Length", str(len(data)))
            self.end_headers()
            self.wfile.write(data)
            return

        # 认证通过后处理请求
        if self.path == "/api":
            data = b'{"message":"hello","status":"ok","items":[1,2,3]}'
            self.send_response(200)
            self.send_header("Content-Type", "application/json; charset=utf-8")
            self.send_header("Content-Length", str(len(data)))
            self.end_headers()
            self.wfile.write()  # TODO, 填写要发送的数据
        else:
            self.send_error(404)

if __name__ == "__main__":
    http.server.HTTPServer(("127.0.0.1", PORT), ).serve_forever() # TODO: 填写处理类


练习6：使用 urllib.request 获取 练习5 带鉴权的 json 内容
- 构造带 Authorization 头的请求获取练习5 的接口
- header: "Authorization": "Bearer <API_KEY_VAR>"
- 处理 HTTPError 并打印服务器返回内容

测试方法：
- 将练习5里的代码复制到一个单独文件 auth-server.py 里.
- 运行这里的代码块，查看输出.
- 修改api-key, 查看不同的api-key的输出结果。


In [ ]:
import json
import urllib.request
import urllib.error

URL = "http://127.0.0.1:8001/api"
api_key = "sk-123"

def main():
    headers = {
        "Authorization": "Bearer {}".format()  # TODO: 填写 api_key 变量
    }

    request = urllib.request.Request(URL, headers=headers, method='GET')
    try:
        resp = urllib.request.urlopen(request)
        raw = resp.read().decode()
    except urllib.error.HTTPError as e:
        # HTTPError 里也有响应体，读取它以查看服务器返回的消息
        raw = e.read().decode()
        print("HTTP error:", e.code, e.reason)

    print(raw)
    try:
        data = json.loads(raw)
        print(data.get("message"))
    except Exception:
        pass

if __name__ == "__main__":
    main()

练习7：把鉴权 key 存到文件中（keys.txt），服务器从文件中读取校验
- 服务器端从 KEYS_FILE 读取所有允许的 key（每行一个）
- 如果文件不存在，选择返回 401
- 鉴权逻辑与练习5相同，只是读取来源不同

测试方法：
- 首先在当前目录下创建 "keys.txt" 文件，在里面写入 sk-123等若干个api_key。
- 运行后不输出任何内容和错误。

In [ ]:
import os
import http.server

PORT = 8001
KEYS_FILE = "keys.txt"

class APIHandler(http.server.BaseHTTPRequestHandler):
  def _get_api_key_from_header(self):
    auth = self.headers.get("Authorization")
    if not auth:
      return None
    parts = auth.split()
    if len(parts) == 2 and parts[0].lower() == "bearer":
      return parts[1]
    return None

  def _key_allowed(self, key):
    if not key:
      return False
    try:
      with open() as f: # TODO: 填写 KEYS_FILE 变量
        keys = {line.strip() for line in f if line.strip()}
      return key in keys
    except FileNotFoundError:
      return False

  def do_GET(self):
    # 鉴权：先检查 headers 里的 Authorization 是否在 keys.txt 中
    api_key = self._get_api_key_from_header()
    if not self._key_allowed(api_key):
      data = "认证不通过".encode("utf-8")
      self.send_response(401)
      self.send_header("Content-Type", "text/plain; charset=utf-8")
      self.send_header("Content-Length", str(len(data)))
      self.end_headers()
      self.wfile.write(data)
      return

    # 认证通过后处理请求
    if self.path == "/api":
      data = b'{"message":"hello","status":"ok","items":[1,2,3]}'
      self.send_response(200)
      self.send_header("Content-Type", "application/json; charset=utf-8")
      self.send_header("Content-Length", str(len(data)))
      self.end_headers()
      self.wfile.write(data)
    else:
      self.send_error(404)

if __name__ == "__main__":
  # 确保 keys.txt 在当前目录（脚本启动目录）可见
  if not os.path.exists(KEYS_FILE):
    pass
  http.server.HTTPServer(("127.0.0.1", PORT), APIHandler).serve_forever()


练习7：将练习6里的代码复制到此处，以测试练习7里的服务器。

测试方法：
- 将练习7里的代码块单独复制到文件 file_server.py 里，运行。
- 执行此处代码，查看输出。
- 修改api_key，查看不同的输出结果。

In [ ]:
# 请补充代码